# Pakistan Housing Market — Exploratory Data Analysis

An in-depth exploration of **16,000+ property listings** from [Zameen.com](https://www.zameen.com/), Pakistan's largest real estate marketplace. This analysis examines pricing patterns, geographic trends, and property characteristics across 12 major Pakistani cities to uncover actionable insights for real estate investors and market analysts.

**Dataset:** Zameen.com Housing Prices (Kaggle)  
**Tools:** Python, pandas, matplotlib, seaborn, plotly

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization defaults
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
df = pd.read_csv('../data/raw/archive_2/zameen.csv', sep='|')
print(f'Dataset: {df.shape[0]:,} listings \u00d7 {df.shape[1]} features')

## 2. Initial Exploration

### 2.1 Sample Listings

In [ ]:
df.head(10)

### 2.2 Data Types & Memory

In [ ]:
df.info()

### 2.3 Numerical Summary

In [ ]:
df.describe()

### 2.4 Categorical Summary

In [ ]:
df.describe(include='object')

### 2.5 First Impressions

- **16,044 listings** across **12 cities** and **2,038 unique locations** — a broad snapshot of Pakistan's urban housing market
- **No missing values** in any column, though data quality issues exist (zero-size properties, extreme outliers)
- **Price range** spans 70,000 PKR to 2.1 Billion PKR — the high end likely includes commercial or incorrectly entered listings
- **Median price** (24M PKR) is roughly half the mean (45.7M PKR), indicating strong right-skew typical of real estate markets
- **Lahore dominates** with the highest listing count, followed by DHA/Bahria Town locations — Pakistan's premium housing societies
- **Size column** contains zero values and an extreme max of 1.2M sq ft — cleaning required before analysis

## 3. Data Quality Investigation

### 3.1 Missing Values Check

In [ ]:
# Count missing values per column and show as both count and percentage
missing_count = df.isnull().sum()
missing_pct = df.isnull().mean() * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('=== Missing Values Summary ===\n')
print(missing_summary.to_string())
print(f'\nTotal cells in dataset: {df.shape[0] * df.shape[1]:,}')
print(f'Total missing cells: {missing_count.sum():,}')

In [ ]:
# Check for "hidden" missing values — zeros that likely represent missing data
print('=== Suspicious Zero Values ===\n')
for col in ['price', 'bedrooms', 'baths', 'size']:
    zero_count = (df[col] == 0).sum()
    zero_pct = (df[col] == 0).mean() * 100
    print(f'{col:>10}: {zero_count:,} zeros ({zero_pct:.2f}%)')

print('\n--- Interpretation ---')
print('A property with 0 size or 0 price is almost certainly a data entry error.')
print('Zero bedrooms could mean a studio/plot, but worth investigating.')

In [ ]:
# Are zero-bedroom and zero-bath listings the same rows?
both_zero = ((df['bedrooms'] == 0) & (df['baths'] == 0)).sum()
only_beds_zero = ((df['bedrooms'] == 0) & (df['baths'] != 0)).sum()
only_baths_zero = ((df['bedrooms'] != 0) & (df['baths'] == 0)).sum()

print('=== Zero Beds vs Zero Baths Overlap ===\n')
print(f'Both bedrooms AND baths = 0:  {both_zero:,}')
print(f'Only bedrooms = 0:            {only_beds_zero:,}')
print(f'Only baths = 0:               {only_baths_zero:,}')

print(f'\n--- Sample of 0-bedroom, 0-bath listings ---\n')
df[(df['bedrooms'] == 0) & (df['baths'] == 0)].sample(10, random_state=42)

### 3.2 Column-by-Column Investigation

In [ ]:
# City distribution — how many listings per city?
print('=== City Distribution ===\n')
city_counts = df['city'].value_counts()
print(city_counts)
print(f'\nTotal unique cities: {df["city"].nunique()}')

In [ ]:
# Location — top 20 most common locations
print('=== Top 20 Locations ===\n')
print(df['location'].value_counts().head(20))
print(f'\nTotal unique locations: {df["location"].nunique()}')

In [ ]:
# Bedrooms distribution
print('=== Bedrooms Distribution ===\n')
print(df['bedrooms'].value_counts().sort_index())
print(f'\nUnique values: {df["bedrooms"].nunique()}')

In [ ]:
# Baths distribution
print('=== Baths Distribution ===\n')
print(df['baths'].value_counts().sort_index())
print(f'\nUnique values: {df["baths"].nunique()}')

In [ ]:
# Price — key percentiles and extreme values
print('=== Price Investigation ===\n')
print(f'Min:    {df["price"].min():>20,.0f} PKR')
print(f'1st %:  {df["price"].quantile(0.01):>20,.0f} PKR')
print(f'5th %:  {df["price"].quantile(0.05):>20,.0f} PKR')
print(f'Median: {df["price"].median():>20,.0f} PKR')
print(f'95th %: {df["price"].quantile(0.95):>20,.0f} PKR')
print(f'99th %: {df["price"].quantile(0.99):>20,.0f} PKR')
print(f'Max:    {df["price"].max():>20,.0f} PKR')

print(f'\n--- Top 10 most expensive listings ---\n')
df.nlargest(10, 'price')[['city', 'location', 'price', 'bedrooms', 'size']]

In [ ]:
# Size — key percentiles and extreme values
print('=== Size Investigation ===\n')
print(f'Min:    {df["size"].min():>15,.0f} sqft')
print(f'1st %:  {df["size"].quantile(0.01):>15,.0f} sqft')
print(f'5th %:  {df["size"].quantile(0.05):>15,.0f} sqft')
print(f'Median: {df["size"].median():>15,.0f} sqft')
print(f'95th %: {df["size"].quantile(0.95):>15,.0f} sqft')
print(f'99th %: {df["size"].quantile(0.99):>15,.0f} sqft')
print(f'Max:    {df["size"].max():>15,.0f} sqft')

print(f'\n--- Listings with 0 size ---\n')
print(f'Count: {(df["size"] == 0).sum()}')

print(f'\n--- Top 10 largest listings ---\n')
df.nlargest(10, 'size')[['city', 'location', 'price', 'bedrooms', 'size']]

### 3.3 Duplicate Check

In [ ]:
# Check for exact duplicate rows
dup_count = df.duplicated().sum()
print(f'=== Duplicate Rows ===\n')
print(f'Exact duplicates: {dup_count:,} ({dup_count/len(df)*100:.2f}%)')

if dup_count > 0:
    print(f'\n--- Sample duplicate rows ---\n')
    # Show a few duplicated rows alongside their originals
    dup_mask = df.duplicated(keep=False)
    print(df[dup_mask].sort_values(by=['city', 'location', 'price']).head(10))

### 3.4 Cardinality Summary

In [ ]:
# Cardinality — how many unique values per column?
print('=== Cardinality (Unique Values per Column) ===\n')
for col in df.columns:
    n_unique = df[col].nunique()
    cardinality = 'LOW' if n_unique < 20 else ('MEDIUM' if n_unique < 100 else 'HIGH')
    print(f'{col:>10}: {n_unique:>6,} unique values  [{cardinality}]')

### 3.5 Data Quality Assessment

**Issues identified for cleaning:**

| # | Issue | Severity | Affected Rows | Cleaning Plan |
|---|-------|----------|---------------|---------------|
| 1 | **32% duplicate rows** (5,174 listings) | HIGH | 5,174 | Drop exact duplicates |
| 2 | **Dirty city name** — `2_FECHS` should be `Islamabad` | LOW | 25 | Remap to Islamabad |
| 3 | **Zero bedrooms/baths** — likely plots (no built structure) | MEDIUM | 1,546 (both zero) | Flag as "plot" property type, keep in dataset |
| 4 | **Zero size** — missing area data disguised as 0 | MEDIUM | 151 | Replace with NaN, decide impute vs drop later |
| 5 | **Extreme prices** — max 2.1B PKR vs median 24M | LOW | ~10 listings >1B | Verify against domain knowledge, cap or keep |
| 6 | **Extreme sizes** — max 1.2M sqft (28 acres) | LOW | ~5 listings >100K | Investigate, likely farmhouses or data errors |

**What's clean:**
- No formal null/NaN values in any column
- Price column is already numeric (no Lakh/Crore strings to parse)
- City names are consistent (except `2_FECHS`)
- Column names are clean and lowercase

## 4. Data Cleaning

### 4.1 Remove Duplicates

In [ ]:
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)
rows_dropped = rows_before - rows_after

print(f'=== Duplicate Removal ===\n')
print(f'Before: {rows_before:,} rows')
print(f'After:  {rows_after:,} rows')
print(f'Dropped: {rows_dropped:,} rows ({rows_dropped/rows_before*100:.1f}%)')

# Reset index after dropping rows
df = df.reset_index(drop=True)
print(f'\nIndex reset: 0 to {len(df)-1}')

### 4.2 Fix City Names

In [ ]:
# Fix dirty city name: 2_FECHS is a housing scheme in Islamabad (E-11 sector)
print('Before fix:')
print(df['city'].value_counts())

df['city'] = df['city'].replace('2_FECHS', 'Islamabad')

print(f'\nAfter fix:')
print(df['city'].value_counts())

In [ ]:
# Strip whitespace from string columns to prevent invisible mismatches
for col in ['city', 'location']:
    before = df[col].nunique()
    df[col] = df[col].str.strip()
    after = df[col].nunique()
    diff = before - after
    print(f'{col}: {before} unique -> {after} unique (merged {diff} whitespace duplicates)')

### 4.3 Convert Data Types

In [ ]:
# Record memory usage before conversion
mem_before = df.memory_usage(deep=True).sum() / 1024
print(f'Memory before: {mem_before:,.1f} KB\n')
print('Current dtypes:')
print(df.dtypes)

In [ ]:
# Convert low-cardinality columns to category dtype
df['city'] = df['city'].astype('category')
df['bedrooms'] = df['bedrooms'].astype('category')
df['baths'] = df['baths'].astype('category')

# Replace zero sizes with NaN (not a real measurement)
df['size'] = df['size'].replace(0, np.nan)

mem_after = df.memory_usage(deep=True).sum() / 1024
savings = (1 - mem_after / mem_before) * 100

print(f'Memory after:  {mem_after:,.1f} KB')
print(f'Savings:       {savings:.1f}%\n')
print('Updated dtypes:')
print(df.dtypes)

### 4.4 Cleaning Verification

In [ ]:
# Final check after cleaning steps
print('=== Post-Cleaning Summary ===\n')
print(f'Shape: {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns')
print(f'Duplicates remaining: {df.duplicated().sum()}')
print(f'Unique cities: {df["city"].nunique()} (was 12, now {df["city"].nunique()} after merging 2_FECHS)')
print(f'Null values in size: {df["size"].isnull().sum()} (converted from zero)')
print(f'\n--- City distribution (cleaned) ---\n')
print(df['city'].value_counts())

print(f'\n--- Data types ---\n')
print(df.dtypes)

print(f'\n--- Quick stats ---')
df.describe()

### 4.5 Cleaning Summary

| Step | Action | Impact |
|------|--------|--------|
| Duplicates | Dropped exact duplicate rows | ~5,174 rows removed (32%) |
| City fix | Remapped `2_FECHS` to `Islamabad` | 25 rows corrected, 12 \u2192 11 cities |
| Whitespace | Stripped leading/trailing spaces from text columns | Prevented invisible mismatches |
| Data types | Converted `city`, `bedrooms`, `baths` to `category` | Reduced memory usage |
| Zero sizes | Replaced 0 values in `size` with NaN | 151 rows now have explicit missing size |